In [1]:
!pip install -q -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 9.1 MB/s eta 0:00:00


In [2]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_6dacc7c207acfb2d686a16c8ac541339"

In [3]:
!kaggle datasets list -s plantdisease

ref                                       title                                              size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------  -------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
mummidi31/plantdisease                    Plantdisease                                      12215  2020-12-14 06:53:29.700000             39          4  0.1875           
ly9802/plantdisease                       PlantDisease                                  855079835  2021-12-10 05:52:35.963000             66          1  0.23529412       
minhaz027/plantdisease                    Plantdisease                                  153905669  2021-04-25 20:53:24.067000             32          2  0.0              
sayanroy058/plantdisease                  PlantDisease                                  934938040  2024-07-23 18:41:45.293000              5     

In [4]:
!kaggle datasets download -d emmarex/plantdisease

Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100%|█████████████████████████████████████████| 658M/658M [00:04<00:00, 159MB/s]



In [5]:
!unzip -q plantdisease.zip

In [ ]:
!ls

In [6]:
import os

dataset_path = "PlantVillage"  # adjust if needed

for cls in os.listdir(dataset_path):
    cls_path = os.path.join(dataset_path, cls)
    if os.path.isdir(cls_path):
        print(cls, len(os.listdir(cls_path)))

Potato___Late_blight 1000
Pepper__bell___Bacterial_spot 997
Tomato__Target_Spot 1404
Tomato_Leaf_Mold 952
Pepper__bell___healthy 1478
Tomato_Late_blight 1909
Potato___healthy 152
Tomato__Tomato_YellowLeaf__Curl_Virus 3209
Tomato_Bacterial_spot 2127
Tomato_healthy 1591
Tomato__Tomato_mosaic_virus 373
Tomato_Spider_mites_Two_spotted_spider_mite 1676
Tomato_Early_blight 1000
Tomato_Septoria_leaf_spot 1771
Potato___Early_blight 1000


In [7]:
!pip install split-folders

In [8]:
import splitfolders

splitfolders.ratio(
    "PlantVillage",
    output="dataset_split",
    seed=42,
    ratio=(0.7, 0.15, 0.15)
)

Copying files: 20639 files [00:02, 7196.24 files/s]


In [9]:
import torch
import torchvision
import torch.nn as nn

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [12]:
# transform = transforms.Compose([
#     transforms.Resize((128,128)),
#     transforms.ToTensor(),
# ])

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1,0.1)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [13]:
train_dataset = datasets.ImageFolder(
    "dataset_split/train",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    "dataset_split/val",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    "dataset_split/test",
    transform=transform
)

In [14]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [15]:
print(train_dataset.classes)

['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [1]:
import json

with open("classes.json", "w") as f:
    json.dump(train_dataset.classes, f)

NameError: name 'train_dataset' is not defined

In [ ]:
print(train_loader.dataset.transform)

In [ ]:
images, labels = next(iter(train_loader))
print(images.shape)

In [ ]:
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
print(images.shape)

In [ ]:
import torch.nn as nn

class PlantDiseaseCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d((1,1)),
        
            nn.Flatten(),
        
            nn.Linear(256, 512),
            nn.ReLU(),
        
            nn.Dropout(0.5),
        
            nn.Linear(512, 15)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = PlantDiseaseCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [ ]:
print(device)

In [ ]:
print(images.shape)

In [ ]:
print(images.device)
print(next(model.parameters()).device)

In [ ]:
images = images.to(device)

print(images.device)
print(next(model.parameters()).device)

In [ ]:
# num_epochs = 40

# for epoch in range(num_epochs):

#     model.train()

#     running_loss = 0.0
#     correct = 0
#     total = 0

#     for images, labels in train_loader:

#         images = images.to(device)
#         labels = labels.to(device)

#         # Reset gradients
#         optimizer.zero_grad()

#         # Forward pass
#         outputs = model(images)

#         # Calculate loss
#         loss = criterion(outputs, labels)

#         # Backpropagation
#         loss.backward()

#         # Update weights
#         optimizer.step()

#         running_loss += loss.item()

#         _, predicted = torch.max(outputs, 1)

#         total += labels.size(0)
#         correct += (predicted == labels).sum().item()

#     train_acc = 100 * correct / total

#     print(
#         f"Epoch [{epoch+1}/{num_epochs}] "
#         f"Loss: {running_loss/len(train_loader):.4f} "
#         f"Train Acc: {train_acc:.2f}%"
#     )

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total

print(f"Test Accuracy: {test_acc:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,10))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
print(train_dataset.classes)

Now USING TRANSFER LEARNING AND SEE IF MY VALIDATION ACCURACY IMPROVES OR NOT


In [11]:
import torch
import torch.nn as nn
from torchvision import models

In [12]:
# from torchvision.models import resnet18, ResNet18_Weights

# weights = ResNet18_Weights.DEFAULT

# model = resnet18(weights=weights)

model = torchvision.models.vgg16(weights="IMAGENET1K_V1")

# model.classifier[6] = nn.Linear(
#     model.classifier[6].in_features,
#     15
# )

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 173MB/s]  


In [13]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [14]:
print(model.classifier[6])

Linear(in_features=4096, out_features=1000, bias=True)


In [15]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)
model = model.to(device)

cuda


In [16]:
# criterion = nn.CrossEntropyLoss()

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     # lr=0.001
#     lr=1e-4
# )

In [23]:
print(train_data.classes)

NameError: name 'train_data' is not defined

In [17]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [18]:
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

Using Feature Extraction first freeze all the layers

In [19]:
train_dataset = datasets.ImageFolder(
    "dataset_split/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "dataset_split/val",
    transform=val_transform
)

test_dataset = datasets.ImageFolder(
    "dataset_split/test",
    transform=val_transform
)

In [21]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [22]:
pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [23]:
from torchinfo import summary

summary(model, input_size=(32, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [32, 1000]                --
├─Sequential: 1-1                        [32, 512, 7, 7]           --
│    └─Conv2d: 2-1                       [32, 64, 224, 224]        1,792
│    └─ReLU: 2-2                         [32, 64, 224, 224]        --
│    └─Conv2d: 2-3                       [32, 64, 224, 224]        36,928
│    └─ReLU: 2-4                         [32, 64, 224, 224]        --
│    └─MaxPool2d: 2-5                    [32, 64, 112, 112]        --
│    └─Conv2d: 2-6                       [32, 128, 112, 112]       73,856
│    └─ReLU: 2-7                         [32, 128, 112, 112]       --
│    └─Conv2d: 2-8                       [32, 128, 112, 112]       147,584
│    └─ReLU: 2-9                         [32, 128, 112, 112]       --
│    └─MaxPool2d: 2-10                   [32, 128, 56, 56]         --
│    └─Conv2d: 2-11                      [32, 256, 56, 56]         29

In [24]:
# for param in model.parameters():
#     param.requires_grad = False
    # Model
model = models.vgg16(weights="DEFAULT")

    # Freeze everything
for param in model.features.parameters():
    param.requires_grad = False

    # Unfreeze selected blocks
blocks = [
    model.features[:5],
    model.features[5:10],
    model.features[10:17],
    model.features[17:24],
    model.features[24:]
    ]

for block in blocks[-1:]:
    for param in block.parameters():
        param.requires_grad = True

In [25]:
num_classes = 15


In [26]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
!pip install optuna

In [75]:
import optuna
import torch.nn as nn
import torch.optim as optim

In [49]:
# model.classifier = nn.Sequential(
#     nn.Linear(25088, 1024),
#     nn.ReLU(),
#     nn.BatchNorm1d(1024),
#     nn.Dropout(0.4),

#     nn.Linear(1024, 512),
#     nn.ReLU(),
#     nn.BatchNorm1d(512),
#     nn.Dropout(0.3),

#     nn.Linear(512, num_classes)
# )

##USING OPTUNA AFTERWARDS


# import torch.nn as nn

# def get_classifier(trial, num_classes):

#     n_layers = trial.suggest_int("n_layers", 1, 4)

#     layers = []
#     in_features = 25088

#     for i in range(n_layers):

#         out_features = trial.suggest_categorical(
#             f"hidden_units_{i}",
#             [128, 256, 512, 1024, 2048]
#         )

#         dropout = trial.suggest_float(
#             f"dropout_{i}",
#             0.2,
#             0.6
#         )

#         layers.extend([
#             nn.Linear(in_features, out_features),
#             nn.ReLU(),
layers = []
in_features = 25088

for _ in range(3):
    layers.extend([
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.BatchNorm1d(256),
        nn.Dropout(0.5)
    ])
    in_features = 256

layers.append(
    nn.Linear(256, 15)
)

model.classifier = nn.Sequential(*layers)
print(model.classifier)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     # lr=0.001
#     lr=1e-4
# )

Sequential(
  (0): Linear(in_features=25088, out_features=256, bias=True)
  (1): ReLU()
  (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=256, out_features=256, bias=True)
  (9): ReLU()
  (10): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): Dropout(p=0.5, inplace=False)
  (12): Linear(in_features=256, out_features=15, bias=True)
)


In [ ]:
# def objective(trial):

#     # Hyperparameters
#     num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
#     neurons_per_layer = trial.suggest_int("neurons_per_layer", 128, 1024, step=128)
#     # epochs = trial.suggest_int("epochs", 10, 50, step=10)
#     learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
#     dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
#     batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
#     optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])
#     weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

#     unfreeze_blocks = trial.suggest_categorical(
#         "unfreeze_blocks",
#         [0, 1, 2]
#     )

#     # Data loaders
#     train_loader = DataLoader(
#         train_dataset,
#         batch_size=batch_size,
#         shuffle=True
#     )

#     val_loader = DataLoader(
#         val_dataset,
#         batch_size=batch_size,
#         shuffle=False
#     )

#     # Model
#     model = models.vgg16(weights="DEFAULT")

#     # Freeze everything
#     for param in model.features.parameters():
#         param.requires_grad = False

#     # Unfreeze selected blocks
#     blocks = [
#         model.features[:5],
#         model.features[5:10],
#         model.features[10:17],
#         model.features[17:24],
#         model.features[24:]
#     ]

#     for block in blocks[-unfreeze_blocks:]:
#         for param in block.parameters():
#             param.requires_grad = True

#     layers = []
#     in_features = 25088

#     for _ in range(num_hidden_layers):
#         layers.extend([
#             nn.Linear(in_features, neurons_per_layer),
#             nn.ReLU(),
#             nn.BatchNorm1d(neurons_per_layer),
#             nn.Dropout(dropout_rate)
#         ])
#         in_features = neurons_per_layer

#     layers.append(
#         nn.Linear(in_features, num_classes)
#     )

#     model.classifier = nn.Sequential(*layers)

#     model = model.to(device)

#     # Optimizer
#     if optimizer_name == "Adam":
#         optimizer = torch.optim.Adam(
#             model.parameters(),
#             lr=learning_rate,
#             weight_decay=weight_decay
#         )

#     elif optimizer_name == "SGD":
#         optimizer = torch.optim.SGD(
#             model.parameters(),
#             lr=learning_rate,
#             momentum=0.9,
#             weight_decay=weight_decay
#         )

#     else:
#         optimizer = torch.optim.RMSprop(
#             model.parameters(),
#             lr=learning_rate,
#             weight_decay=weight_decay
#         )

#     criterion = nn.CrossEntropyLoss()

#     # =====================
#     # TRAINING
#     # =====================

#     for epoch in range(15):

#         model.train()

#         for images, labels in train_loader:

#             images = images.to(device)
#             labels = labels.to(device)

#             optimizer.zero_grad()

#             outputs = model(images)

#             loss = criterion(outputs, labels)

#             loss.backward()

#             optimizer.step()

#     # =====================
#     # VALIDATION
#     # =====================

#     model.eval()

#     correct = 0
#     total = 0

#     with torch.no_grad():

#         for images, labels in val_loader:

#             images = images.to(device)
#             labels = labels.to(device)

#             outputs = model(images)

#             _, predicted = torch.max(outputs, 1)

#             total += labels.size(0)
#             correct += (predicted == labels).sum().item()

#     test_acc = 100 * correct / total

    
#     return test_acc

In [ ]:
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=10)

In [48]:
from torchinfo import summary
summary(model, input_size=(32, 3, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [32, 15]                  --
├─Sequential: 1-1                        [32, 512, 7, 7]           --
│    └─Conv2d: 2-1                       [32, 64, 224, 224]        (1,792)
│    └─ReLU: 2-2                         [32, 64, 224, 224]        --
│    └─Conv2d: 2-3                       [32, 64, 224, 224]        (36,928)
│    └─ReLU: 2-4                         [32, 64, 224, 224]        --
│    └─MaxPool2d: 2-5                    [32, 64, 112, 112]        --
│    └─Conv2d: 2-6                       [32, 128, 112, 112]       (73,856)
│    └─ReLU: 2-7                         [32, 128, 112, 112]       --
│    └─Conv2d: 2-8                       [32, 128, 112, 112]       (147,584)
│    └─ReLU: 2-9                         [32, 128, 112, 112]       --
│    └─MaxPool2d: 2-10                   [32, 128, 56, 56]         --
│    └─Conv2d: 2-11                      [32, 256, 56, 56]   

In [51]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [52]:


model = model.to(device)

In [53]:
print(type(model))

<class 'torchvision.models.vgg.VGG'>


In [54]:
print(next(model.parameters()).device)

cuda:0


In [55]:
print(model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [86]:
all_labels = []

for _, labels in train_loader:
    all_labels.extend(labels.tolist())

print("Min:", min(all_labels))
print("Max:", max(all_labels))
print("Unique:", sorted(set(all_labels)))

Min: 0
Max: 14
Unique: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


In [56]:
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.0001
)


In [34]:
for images, labels in train_loader:

    images = images.to(device)

    outputs = model(images)

    print(outputs.shape)

    break

torch.Size([16, 15])


In [35]:
print(optimizer.param_groups[0]["lr"])

0.0001


In [57]:
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(trainable)

13639183


In [37]:
print(model.classifier)

Sequential(
  (0): Linear(in_features=25088, out_features=256, bias=True)
  (1): ReLU()
  (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=256, out_features=256, bias=True)
  (9): ReLU()
  (10): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): Dropout(p=0.5, inplace=False)
  (12): Linear(in_features=256, out_features=15, bias=True)
)


In [38]:
images, labels = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print(outputs[:2])

tensor([[ 0.5126,  0.5090,  0.5717, -0.2749,  0.3825, -0.7528, -0.1316,  0.3907,
          0.0832, -0.7178, -1.0115,  0.2263,  1.3310, -0.5986, -0.0791],
        [-0.4497, -0.0079,  0.4841, -1.1376, -0.2692,  1.2715, -0.4662,  0.3747,
          0.2415,  0.6656,  0.8862,  0.4391, -0.1021,  0.1540, -0.2641]],
       device='cuda:0')


In [91]:
images, labels = next(iter(train_loader))

print(labels[:20])

tensor([ 2, 10, 10,  7,  0, 12,  2,  1, 12,  0,  3,  3, 10,  6, 10, 10])


In [92]:
with torch.no_grad():
    outputs = model(images.to(device))
    preds = outputs.argmax(1)

print(preds[:20].cpu())

tensor([ 8, 14, 11, 10, 14,  2,  1,  9, 12,  1,  5,  8,  6, 12,  4,  2])


In [39]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

14440
3089
3109


AttributeError: 'ImageFolder' object has no attribute 'shape'

In [58]:

for epoch in range(15):

    # =====================
    # TRAINING
    # =====================
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
      

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total








 

   



    print(
        f"Epoch [{epoch+1}/15] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% | "
        
    )


Epoch [1/15] Train Loss: 1.4821 Train Acc: 55.87% | 


KeyboardInterrupt: 

In [58]:
print(model.classifier)

trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)
print(trainable)

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)
138357544


In [ ]:
# The validation accuracy of Resnet - 95.92%
# The validation accuracy of VGG16 - 94.33%
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total

print(f"Test Accuracy: {test_acc:.2f}%")

In [ ]:
# The validation accuracy of Resnet - 95.92%
# The validation accuracy of VGG16 - 94.33%
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total

print(f"Test Accuracy: {test_acc:.2f}%")

Now trying Vgg16 

In [36]:
# After training finishes
torch.save(model.state_dict(), "vgg16_plant_disease.pth")

print("Model saved successfully!")

Model saved successfully!


In [46]:
# model.eval()

# image, label = test_dataset[900]

# with torch.no_grad():
#     output = model(image.unsqueeze(0).to(device))
#     pred = output.argmax(1)

# print("Actual:", test_dataset.classes[label])
# print("Predicted:", test_dataset.classes[pred.item()])

Actual: Tomato_Bacterial_spot
Predicted: Tomato_Late_blight


In [47]:
# correct = 0

# for i in range(100):
#     image, label = test_dataset[i]

#     with torch.no_grad():
#         output = model(image.unsqueeze(0).to(device))
#         pred = output.argmax(1).item()

#     if pred == label:
#         correct += 1

# print("Correct:", correct)
# print("Accuracy:", correct/100)

Correct: 0
Accuracy: 0.0


In [ ]:
# import os

# for f in os.listdir("/kaggle/working"):
#     print(f)

In [ ]:
# from IPython.display import FileLink

# FileLink('/kaggle/working/vgg16_plant_disease.pth')

In [ ]:
# import os

# size_mb = os.path.getsize("/kaggle/working/vgg16_plant_disease.pth") / (1024*1024)
# print(f"{size_mb:.2f} MB")

In [ ]:
# !ls -lh /kaggle/working/*.pth

In [ ]:
# !du -h /kaggle/working/*.pth

In [ ]:
# !mkdir -p /kaggle/working/model_dataset
# !cp /kaggle/working/vgg16_plant_disease.pth /kaggle/working/model_dataset/

In [ ]:
# !mkdir -p /kaggle/working/model_upload
# !cp /kaggle/working/vgg16_plant_disease.pth /kaggle/working/model_upload/
# !ls -lh /kaggle/working/model_upload

In [ ]:
# !ls -lh /kaggle/working/model_upload.zip

In [ ]:
# !zip model.zip vgg16_plant_disease.pth

In [ ]:
# !ls -lh model.zip

In [48]:
# print("Train Classes:")
# print(train_dataset.classes)

# print("\nTest Classes:")
# print(test_dataset.classes)

Train Classes:
['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']

Test Classes:
['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [ ]:
# !ls -lh /kaggle/working

In [57]:
# print(model.classifier)

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)


In [50]:
# for i in range(10):

#     image, label = test_dataset[i]

#     with torch.no_grad():
#         output = model(image.unsqueeze(0).to(device))
#         pred = output.argmax(1).item()

#     print(
#         f"Actual: {label}  Pred: {pred}"
#     )

Actual: 0  Pred: 7
Actual: 0  Pred: 9
Actual: 0  Pred: 7
Actual: 0  Pred: 6
Actual: 0  Pred: 6
Actual: 0  Pred: 6
Actual: 0  Pred: 7
Actual: 0  Pred: 7
Actual: 0  Pred: 3
Actual: 0  Pred: 6


In [ ]:
# !du -h model.zip

In [51]:
# model.load_state_dict(torch.load(...))

AttributeError: 'ellipsis' object has no attribute 'seek'. You can only torch.load from a file that is seekable. Please pre-load the data into a buffer like io.BytesIO and try to load from it instead.

In [52]:
# checkpoint = torch.load(
#     "/kaggle/working/vgg16_plant_disease.pth",
#     map_location="cpu"
# )

# print(type(checkpoint))
# print(len(checkpoint))
# print(list(checkpoint.keys())[:10])

<class 'collections.OrderedDict'>
49
['features.0.weight', 'features.0.bias', 'features.2.weight', 'features.2.bias', 'features.5.weight', 'features.5.bias', 'features.7.weight', 'features.7.bias', 'features.10.weight', 'features.10.bias']


In [53]:
# model = models.vgg16(weights=None)

# # build classifier here

# model.load_state_dict(
#     torch.load(
#         "/kaggle/working/vgg16_plant_disease.pth",
#         map_location=device
#     )
# )

# model.eval()

# correct = 0

# for i in range(100):

#     image, label = test_dataset[i]

#     with torch.no_grad():
#         pred = model(
#             image.unsqueeze(0).to(device)
#         ).argmax(1).item()

#     if pred == label:
#         correct += 1

# print(correct)

RuntimeError: Error(s) in loading state_dict for VGG:
	Missing key(s) in state_dict: "classifier.3.weight", "classifier.3.bias". 
	Unexpected key(s) in state_dict: "classifier.8.weight", "classifier.8.bias", "classifier.10.weight", "classifier.10.bias", "classifier.10.running_mean", "classifier.10.running_var", "classifier.10.num_batches_tracked", "classifier.12.weight", "classifier.12.bias", "classifier.2.weight", "classifier.2.bias", "classifier.2.running_mean", "classifier.2.running_var", "classifier.2.num_batches_tracked", "classifier.4.weight", "classifier.4.bias", "classifier.6.running_mean", "classifier.6.running_var", "classifier.6.num_batches_tracked". 
	size mismatch for classifier.0.weight: copying a param with shape torch.Size([256, 25088]) from checkpoint, the shape in current model is torch.Size([4096, 25088]).
	size mismatch for classifier.0.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([4096]).
	size mismatc